# Combining Two Ukrainian Datasets

**Note:** We combine two Ukrainian-language datasets.  
The first consists of approximately **1,000 Ukrainian comments** collected from X (formerly Twitter) using **Apify**.  
The second dataset, sourced from **Hugging Face**, contains **5,000 Twitter comments** (2,500 toxic and 2,500 non-toxic) and was originally used in a **study on Ukrainian binary toxicity classification** ([link](https://huggingface.co/datasets/ukr-detect/ukr-toxicity-dataset)). This dataset was introduced in the paper [*“Toxicity Classification in Ukrainian”*](https://aclanthology.org/2024.woah-1.19.pdf).

## Importing Dependencies

In [2]:
import pandas as pd
from datasets import load_dataset

from toxicity_detector.config.paths import UKR_RAW
from toxicity_detector.config.labels import LABELS_EN

c:\Users\Amina\Desktop\Thesis project\PROJECT\my_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Loading Data

In [3]:
df = pd.read_csv(UKR_RAW["comments"])
hf_dataset = load_dataset("ukr-detect/ukr-toxicity-dataset")

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1039 entries, 0 to 1038
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   comment        1039 non-null   object
 1   toxic          1039 non-null   int64 
 2   severe_toxic   1039 non-null   int64 
 3   obscene        1039 non-null   int64 
 4   threat         1039 non-null   int64 
 5   insult         1039 non-null   int64 
 6   identity_hate  1039 non-null   int64 
dtypes: int64(6), object(1)
memory usage: 56.9+ KB


In [5]:
hf_df = hf_dataset["train"].to_pandas()
hf_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    5000 non-null   object
 1   toxic   5000 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 78.2+ KB


In [6]:
hf_df = hf_df.rename(columns={"text": "comment"})

In [7]:
for col in LABELS_EN[1:]:
    hf_df[col] = 0
    
hf_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   comment        5000 non-null   object
 1   toxic          5000 non-null   int64 
 2   severe_toxic   5000 non-null   int64 
 3   obscene        5000 non-null   int64 
 4   threat         5000 non-null   int64 
 5   insult         5000 non-null   int64 
 6   identity_hate  5000 non-null   int64 
dtypes: int64(6), object(1)
memory usage: 273.6+ KB


In [11]:
hf_df[["comment"]].sample(10)

,comment
84,"Гаразд, вмирала би, з'їла би снікерс."
1134,"Абідна коли відписуються улюблені твіттерські,..."
3199,"Нерви спочатку капєц класно цікаво, а потім: ""..."
3156,"Жінко, просто пизди себе щоразу, як проскакуют..."
2296,"Була весь день схарена, бо на весь Берлін нині..."
778,"реплаї засирають стрічку твоїм читачам, які мо..."
729,"як пояснити татові, що в мене видно шкарпетки ..."
1477,Уявна республіка Між | Збруч: - чудовий есей від
1437,перенесла зустріч на середу. пообіцяла зустріт...
2067,Тре триматися від усіх на відстані.


## Combining Datasets

In [7]:
def combine_datasets(df1, df2, output_path):    
    df = pd.concat([df1, df2], ignore_index=True)
    
    df.to_csv(output_path, index=False)

In [8]:
# Combining datasets with only comments
combine_datasets(df[["comment"]], hf_df[["comment"]], UKR_RAW["combined"])

In [12]:
# Combining datasets with only labels
combine_datasets(df[LABELS_EN], hf_df[LABELS_EN], UKR_RAW["labels"])